# Adam — A Method for Stochastic Optimization (2014)

_Papers · Foundations & Optimization_

**Per-parameter step sizes from running averages of the gradient and its square — so most networks train well with almost no learning-rate tuning.**

---

This notebook is a practice scaffold for the **Adam — A Method for Stochastic Optimization (2014)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch, torch.nn as nn

torch.manual_seed(0)

class MyAdam:
    """Adam from scratch — Algorithm 1 of Kingma & Ba (2014)."""
    def __init__(self, params, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8):
        self.params = list(params)
        self.lr, self.b1, self.b2, self.eps = lr, beta1, beta2, eps
        self.m = [torch.zeros_like(p) for p in self.params]   # 1st moment (init 0)
        self.v = [torch.zeros_like(p) for p in self.params]   # 2nd moment (init 0)
        self.t = 0

    @torch.no_grad()
    def step(self):
        self.t += 1                                           # t <- t + 1
        for i, p in enumerate(self.params):
            g = p.grad
            self.m[i] = self.b1*self.m[i] + (1-self.b1)*g     # m_t  (Alg.1 line: biased 1st moment)
            self.v[i] = self.b2*self.v[i] + (1-self.b2)*g*g   # v_t  (biased 2nd raw moment)
            mhat = self.m[i] / (1 - self.b1**self.t)          # bias-corrected 1st moment
            vhat = self.v[i] / (1 - self.b2**self.t)          # bias-corrected 2nd moment
            p -= self.lr * mhat / (vhat.sqrt() + self.eps)    # theta update (eps OUTSIDE sqrt)

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()

# ---- THE ORACLE: MyAdam must equal torch.optim.Adam over several steps ----
w_mine = torch.randn(5, 3, requires_grad=True)
w_ref  = w_mine.detach().clone().requires_grad_(True)        # identical start
opt_mine = MyAdam([w_mine], lr=1e-2)
opt_ref  = torch.optim.Adam([w_ref], lr=1e-2)               # same default betas=(0.9,0.999), eps=1e-8

X = torch.randn(20, 5); target = torch.randn(20, 3)
for _ in range(6):
    opt_mine.zero_grad()
    (((X @ w_mine) - target)**2).mean().backward(); opt_mine.step()
    opt_ref.zero_grad()
    (((X @ w_ref) - target)**2).mean().backward(); opt_ref.step()

print("allclose vs torch.optim.Adam after 6 steps:",
      torch.allclose(w_mine, w_ref, atol=1e-7))            # expect True
print("max abs diff:", (w_mine - w_ref).abs().max().item())  # ~7e-9

# ---- recompute the one-step worked example: theta0=0, g=0.1, t=1, defaults ----
th = torch.zeros(1, requires_grad=False)
o = MyAdam([th], lr=0.001)            # alpha=0.001, b1=0.9, b2=0.999, eps=1e-8
th.grad = torch.tensor([0.1])
o.step()
print("worked example: m1=0.01, v1=1e-5, mhat=0.1, vhat=0.01, step=-0.001")
print("MyAdam new theta:", round(th.item(), 8))            # -0.001

## Visualize the data & results

_Minimize the same small, badly-conditioned least-squares loss from the same start with plain SGD vs Adam (each at a sensible learning rate) — does Adam's per-parameter step reach a much lower loss in the same number of steps?_

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

# tiny ill-conditioned least-squares: features scaled unevenly -> one SGD lr struggles
N, D = 200, 6
scales = np.array([4.0, 3.0, 2.0, 1.0, 0.5, 0.25])
X = rng.normal(0, 1, (N, D)) * scales
w_true = rng.normal(0, 1, D)
y = X @ w_true + rng.normal(0, 0.01, N)

def loss_grad(w):
    r = X @ w - y
    return 0.5*np.mean(r**2), (X.T @ r)/N

def run_sgd(lr, steps=60):
    w = np.zeros(D); out = []
    for _ in range(steps):
        L, g = loss_grad(w); out.append(L); w -= lr*g
    return out

def run_adam(lr, steps=60, b1=0.9, b2=0.999, eps=1e-8):
    w = np.zeros(D); m = np.zeros(D); v = np.zeros(D); out = []
    for t in range(1, steps+1):
        L, g = loss_grad(w); out.append(L)
        m = b1*m + (1-b1)*g                 # first moment
        v = b2*v + (1-b2)*g*g               # second moment
        mh = m/(1-b1**t); vh = v/(1-b2**t)  # bias correction
        w -= lr*mh/(np.sqrt(vh)+eps)        # per-parameter update
    return out

sgd  = run_sgd(lr=0.04)
adam = run_adam(lr=0.1)
print("SGD  final loss:", round(sgd[-1], 4))    # ~0.1131
print("Adam final loss:", round(adam[-1], 4))   # ~0.0019

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Do one Adam step with paper defaults from $\theta_0=0$ but a much larger gradient $g_1=10$ at $t=1$. How big is the step, and what does that tell you?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- First moment: $m_1=(1-0.9)\cdot 10 = 1.0$. — _Smoothed gradient._
- Second moment: $v_1=(1-0.999)\cdot 10^2 = 0.001\cdot 100 = 0.1$. — _Average squared gradient._
- Bias-correct: $\hat m_1=1.0/0.1=10$, $\hat v_1=0.1/0.001=100$. — _Undo zero-init bias at $t=1$._
- Step: $0.001\cdot 10/(\sqrt{100}+\epsilon)=0.001\cdot 10/10=0.001$. New $\theta_1=-0.001$. — _Direction over size._

**Answer:** The step is again $-0.001=-\alpha$, identical to the $g_1=0.1$ case. Because Adam divides the smoothed gradient by its own recent magnitude, the first step is $\pm\alpha$ whether the gradient is 0.1 or 10. This scale-invariance is why one learning rate works across very different weights.

</details>

**Problem 2.** Why does Adam typically beat plain SGD on a badly-conditioned problem (some directions much steeper than others)?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- On steep directions $g^2$ is large, so $\sqrt{\hat v}$ is large and the step there shrinks. — _Prevents overshoot/oscillation in steep directions._
- On flat directions $g^2$ is small, so $\sqrt{\hat v}$ is small and the step there grows. — _Makes progress where SGD would crawl._
- SGD must use one global rate that is safe for the steepest direction, so it crawls in the flat ones. — _Single rate is a compromise._

**Answer:** Adam normalizes each weight's step by that weight's recent gradient size, so it can take large steps in flat directions and small steps in steep ones simultaneously. Plain SGD, with one rate, must keep that rate small enough for the steepest direction and therefore crawls in the flat ones. In our CODEVIZ run Adam reaches loss ~0.0019 vs SGD's ~0.11 in 60 steps.

</details>

**Problem 3.** Ablation: in the CODE, remove the bias correction (use $m_t,v_t$ directly instead of $\hat m_t,\hat v_t$) and rerun the allclose against torch.optim.Adam. What happens, and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Replace $\hat m_t/(\sqrt{\hat v_t}+\epsilon)$ with $m_t/(\sqrt{v_t}+\epsilon)$. — _Drops the $1/(1-\beta^{\,t})$ factors._
- Run the same 6 steps and call $\texttt{torch.allclose}$. — _PyTorch keeps the correction._
- Inspect the first step's size. — _Early steps are where the bias is largest._

**Answer:** The allclose now returns False. Without correction, at small $t$ the moments are biased toward zero — but $\hat v_t$ would have been divided by the tiny $1-\beta_2^{\,t}$, so dropping it makes the denominator $\sqrt{v_t}$ far too small early on, giving much larger early steps than PyTorch. The mismatch is largest at step 1 and shrinks as $t$ grows (the correction factor approaches 1). This confirms the bias correction is essential to match Adam exactly.

</details>